In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from itertools import product
import sys
import pandas as pd
from torch.utils.data import DataLoader
!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 216 (delta 4), reused 6 (delta 2), pack-reused 207 (from 1)
Receiving objects: 100% (216/216), 2.81 MiB | 41.08 MiB/s, done.
Resolving deltas: 100% (115/115), done.
db01887 (HEAD -> main, origin/main, origin/HEAD) float32


### Import necessary functions/modules from git repository

In [ ]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
from dataset import LSTM_Dataset
import lstm
if torch.cuda.is_available():
    device = torch.device("cuda")

In [5]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### load device, datasets, config into environment.

In [6]:
final_short_window_lstm = lstm.LSTMBaseline(hidden_size = 128, num_layers = 1).to(device)
final_long_window_lstm = lstm.LSTMBaseline(hidden_size = 256, num_layers = 1).to(device)

In [7]:
final_short_window_lstm.load_state_dict(torch.load("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection/saved_model_weights/LSTM_short_window_weights.pth", weights_only=True))
final_long_window_lstm.load_state_dict(torch.load("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection/saved_model_weights/LSTM_long_window_weights.pth", weights_only=True))

<All keys matched successfully>

### Load validation, and testing datasets for both models

In [8]:
forecasting_validation_dataset_s = LSTM_Dataset(device, 5, start = 1000000, end = 3500000, train = False)
forecasting_testing_dataset_s = LSTM_Dataset(device, 5, start = 3500000, end = 5000000, train = False)
forecasting_validation_dataset_l = LSTM_Dataset(device, 100, start = 1000000, end = 3500000, train = False)
forecasting_testing_dataset_l = LSTM_Dataset(device, 100, start = 3500000, end = 5000000, train = False)

In [ ]:
def evaluate_lstm_scores(model, dataset, dist):
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    scores, labels, categories = [], [], []
    model.eval()
    with torch.no_grad():
        for X, y, label, category in loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            error = (y - y_pred).view(len(X), -1).cpu().numpy()
            score = -dist.logpdf(error)  #high = anomaly
            scores.extend(score)
            labels.extend(label.numpy())
            categories.extend(category)
    return np.array(scores), np.array(labels), np.array(categories)

scores_s, labels_s, categories_s = evaluate_lstm_scores(final_short_window_lstm, forecasting_validation_dataset_s)
scores_l, labels_l, categories_l = evaluate_lstm_scores(final_long_window_lstm, forecasting_validation_dataset_l)

### Tuning the anomaly smoothing window size on separate validation data that contains anomalies

In [41]:
# Assuming 3.8% contamination rate from the dataset description
window_grid = [210, 240, 270, 300, 330, 360, 390, 420, 450, 480]
def tune_anomaly_window_size(scores, labels, window_grid, forecasting_validation_dataset):
    for window in window_grid:
        if window >= 1:
            current_score = np.convolve(scores, np.ones(window) / window, mode='same')
        threshold = np.percentile(current_score, 100 - 3.8)
        TP = np.sum((current_score > threshold) & (labels == 1))
        FP = np.sum((current_score > threshold) & (labels == 0))
        recall = TP/forecasting_validation_dataset.total_anomalies
        precision = TP/(TP + FP)
        F1 = 2 * (precision * recall)/(precision + recall)
        print(f"| window: {window} | recall {recall:.2f}, precision {precision:.2f}, F1 {F1:.4f}")
print("Short Window:")
tune_anomaly_window_size(scores_s, labels_s, window_grid, forecasting_validation_dataset_s)
print("\nLong Window: ")
tune_anomaly_window_size(scores_l, labels_l, window_grid, forecasting_validation_dataset_l)

Short Window:
| window: 210 | recall 0.42, precision 0.56, F1 0.4792
| window: 240 | recall 0.42, precision 0.56, F1 0.4843
| window: 270 | recall 0.43, precision 0.57, F1 0.4901
| window: 300 | recall 0.43, precision 0.57, F1 0.4936
| window: 330 | recall 0.43, precision 0.58, F1 0.4954
| window: 360 | recall 0.44, precision 0.58, F1 0.4968
| window: 390 | recall 0.44, precision 0.58, F1 0.4966
| window: 420 | recall 0.44, precision 0.58, F1 0.4968
| window: 450 | recall 0.44, precision 0.58, F1 0.4990
| window: 480 | recall 0.44, precision 0.58, F1 0.4990

Long Window: 
| window: 210 | recall 0.42, precision 0.56, F1 0.4802
| window: 240 | recall 0.43, precision 0.57, F1 0.4887
| window: 270 | recall 0.44, precision 0.58, F1 0.4965
| window: 300 | recall 0.44, precision 0.59, F1 0.5056
| window: 330 | recall 0.45, precision 0.59, F1 0.5110
| window: 360 | recall 0.45, precision 0.60, F1 0.5122
| window: 390 | recall 0.45, precision 0.60, F1 0.5162
| window: 420 | recall 0.46, precisi

#### window for short: 450<br>window for long: 420

### Evaluating on the test datasets

In [ ]:
test_scores_s, test_labels_s, test_categories_s = evaluate_lstm_scores(final_short_window_lstm, forecasting_testing_dataset_s)
test_scores_l, test_labels_l, test_categories_l = evaluate_lstm_scores(final_long_window_lstm, forecasting_testing_dataset_l)

In [39]:
def evaluate_test_F1(scores, window, test_labels, dataset):
    current_test_score = np.convolve(scores, np.ones(window) / window, mode='same')
    threshold = np.percentile(current_test_score, 100 - 3.8)
    TP = np.sum((current_test_score > threshold) & (test_labels == 1))
    FP = np.sum((current_test_score > threshold) & (test_labels == 0))
    recall = TP/dataset.total_anomalies
    precision = TP/(TP + FP)
    F1 = 2 * (precision * recall)/(precision + recall)
    print(f"| evaluation_threshold: {threshold:.4f} | window: {window} | recall {recall:.4f}, precision {precision:.4f}, F1 {F1:.4f}")
print("Short Window:")
evaluate_test_F1(test_scores_s, 450, test_labels_s, forecasting_testing_dataset_s)
print("Long Window:")
evaluate_test_F1(test_scores_l, 420, test_labels_l, forecasting_testing_dataset_l)

Short Window:
| evaluation_threshold: 0.4859 | window: 450 | recall 0.5282, precision 0.5949, F1 0.5596
Long Window:
| evaluation_threshold: 0.2498 | window: 420 | recall 0.5481, precision 0.6174, F1 0.5807


In [38]:
def generate_df(test_scores, test_categories, window):
    series = []
    current_test_score = np.convolve(test_scores, np.ones(window)/window, mode='same')
    threshold = np.percentile(current_test_score, 100 - 3.8)
    cat_dict = {}
    normal_mask = test_categories == 0
    FPR = np.sum((current_test_score[normal_mask] > threshold)) / np.sum(normal_mask)
    cat_dict["FPR (0.0)"] = np.round(FPR, decimals=2).item()

    for cat in np.unique(test_categories):
        if cat == 0.0:
            continue
        cat_mask = test_categories == cat
        cat_TP = np.sum((current_test_score[cat_mask] > threshold))
        recall = cat_TP / np.sum(cat_mask)
        cat_dict[f"recall ({cat})"] = np.round(recall, decimals=2).item()

    series.append(pd.Series(cat_dict, name=f"thresh={threshold:.2f}"))
    
    f_concat = pd.concat(series, axis=1)
    return f_concat
pd.concat([generate_df(test_scores_s, test_categories_s, 450), 
           generate_df(test_scores_l, test_categories_l, 420)], axis=1, keys=["Short Window", "Long Window"])

,Short Window,Long Window
,thresh=0.49,thresh=0.25
FPR (0.0),0.02,0.02
recall (1.0),0.95,0.96
recall (2.0),0.83,0.81
recall (3.0),0.75,0.76
recall (4.0),0.34,0.31
recall (5.0),0.15,0.22
recall (6.0),0.00,0.00
recall (7.0),0.00,0.09
recall (8.0),0.71,0.77


In [45]:
with torch.no_grad():
    for X, y, _, _ in DataLoader(forecasting_validation_dataset_l, batch_size=512):
        pred = final_long_window_lstm(X)
        last_step = X[:, -1, :]
        
        corr = np.corrcoef(pred.cpu().numpy().flatten(), 
                           last_step.cpu().numpy().flatten())[0,1]
        print(f"Correlation between prediction and last input step: {corr:.4f}")
        break

Correlation between prediction and last input step: 0.9987
